In [1]:
# Szükséges importok
import ipywidgets as widgets
from IPython.display import display, clear_output
import h5py
import os
import glob
import numpy as np

# --- Fájl tab: HDF5 fájl elérési út és név megadása, csak beolvasás gomb ---
import os
import ipywidgets as widgets
from IPython.display import clear_output
import h5py

file_output = widgets.Output()

directory_widget = widgets.Text(
    value=os.getcwd(),
    description="Mappa:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
filename_widget = widgets.Text(
    value="scatter_data.h5",
    description="Fájlnév:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

def get_hdf5_path():
    return os.path.join(directory_widget.value, filename_widget.value)

file_load_button = widgets.Button(description="Beolvasás", button_style='success')

loaded_params = {}

def load_hdf5_file(b):
    with file_output:
        clear_output(wait=True)
        path = get_hdf5_path()
        if not os.path.exists(path):
            print("A megadott fájl nem található.")
            return
        try:
            with h5py.File(path, "r") as f:
                params = {}
                for k, v in f.attrs.items():
                    params[k] = v
                if "params" in f:
                    for k, v in f["params"].attrs.items():
                        params[k] = v
                loaded_params.clear()
                loaded_params.update(params)
                print(f"Sikeresen beolvasva: {path}")
        except Exception as e:
            print(f"Hiba a fájl beolvasásakor: {e}")
    # Paraméter tab frissítése a beolvasás után
    update_param_tab()

file_load_button.on_click(load_hdf5_file)

file_tab = widgets.VBox([
    widgets.Label("HDF5 fájl elérési út és név megadása"),
    directory_widget,
    filename_widget,
    file_load_button,
    file_output
])

In [2]:
# --- Paraméterek tab: strukturált megjelenítés és plotok ---
import matplotlib.pyplot as plt

def create_param_widgets():
    # Külön szedjük a paramétereket
    grid_keys = ['uxgrid_num', 'uxgrid_dx', 'uxgrid_width']
    unit_keys = ['hbar', 'm0', 'kappa0', 'e0']
    pot_keys = ['wallwidth', 'wallheight', 'wallrise', 'wellwidth', 'welldepth', 'wellfall', 'num_cells', 'cell_spacing']
    disp_keys = ['disp_grid_mode', 'disp_energy_min', 'disp_energy_max', 'disp_number_of_points']

    grid_items = []
    unit_items = []
    pot_items = []
    disp_items = []
    # other_items = []  # NE gyűjtsük az egyéb paramétereket

    for k, v in loaded_params.items():
        if k in grid_keys:
            grid_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))
        elif k in unit_keys:
            unit_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))
        elif k in pot_keys:
            # pot_items.append(...)  # Potenciál paraméterek nem jelennek meg szövegesen
            continue
        elif k in disp_keys:
            disp_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))
        # else:  # Egyéb paraméterek nem kellenek
        #     continue

    # Potenciál plot visszaállítása
    pot_plot_output = widgets.Output()
    with pot_plot_output:
        pot_plot_output.clear_output(wait=True)
        try:
            # Paraméterek kinyerése
            import numpy as np
            wallwidth = float(loaded_params.get('wallwidth', 3.0))
            wallheight = float(loaded_params.get('wallheight', 1.0))
            wallrise = float(loaded_params.get('wallrise', 1.0))
            wellwidth = float(loaded_params.get('wellwidth', 10.0))
            welldepth = float(loaded_params.get('welldepth', 1.0))
            wellfall = float(loaded_params.get('wellfall', 1.0))
            num_cells = int(loaded_params.get('num_cells', 1))
            cell_spacing = float(loaded_params.get('cell_spacing', 0.0))
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            # Egyszerű potenciál plot (szimmetrikus, nem a quantum_toolkit függvényeivel)
            pot = np.zeros_like(uxgrid)
            cell_length = 2*wallwidth + wellwidth + cell_spacing
            for i in range(num_cells):
                offset = (i - (num_cells-1)/2) * cell_length
                # Falak
                left = offset - (wellwidth/2 + wallwidth)
                right = offset + (wellwidth/2 + wallwidth)
                well_left = offset - wellwidth/2
                well_right = offset + wellwidth/2
                pot[(uxgrid >= left) & (uxgrid < well_left)] = wallheight
                pot[(uxgrid > well_right) & (uxgrid <= right)] = wallheight
                pot[(uxgrid >= well_left) & (uxgrid <= well_right)] = -welldepth
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, pot)
            plt.xlabel("x")
            plt.ylabel("Potential")
            plt.title("Potenciál (egyszerűsített)")
            plt.grid()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Potenciál plot hiba:", e)

    disp_plot_output = widgets.Output()
    # Diszperzió plot
    with disp_plot_output:
        disp_plot_output.clear_output(wait=True)
        try:
            disp_mode = loaded_params.get('disp_grid_mode', 'unified in k')
            e_min = float(loaded_params.get('disp_energy_min', 0.05))
            e_max = float(loaded_params.get('disp_energy_max', 2.0))
            npts = int(loaded_params.get('disp_number_of_points', 10))
            hbar = float(loaded_params.get('hbar', 1.0))
            m0 = float(loaded_params.get('m0', 8.8))
            # Egyszerű parabola diszperzió
            e_vals = np.linspace(e_min, e_max, npts)
            k_vals = np.sqrt(2*m0*e_vals)/hbar
            plt.figure(figsize=(6,2.5))
            if disp_mode == 'unified in k':
                plt.plot(k_vals, e_vals, 'o-')
                plt.xlabel("k")
                plt.ylabel("energy")
                plt.title("Diszperzió (egyszerűsített, k szerint)")
            else:
                plt.plot(e_vals, k_vals, 'o-')
                plt.xlabel("energy")
                plt.ylabel("k")
                plt.title("Diszperzió (egyszerűsített, energy szerint)")
            plt.grid()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Diszperzió plot hiba:", e)

    sections = []
    if grid_items:
        sections.append(widgets.HTML("<b>Grid paraméterek</b>"))
        sections.append(widgets.VBox(grid_items))
    if unit_items:
        sections.append(widgets.HTML("<b>Unit system paraméterek</b>"))
        sections.append(widgets.VBox(unit_items))
    # Potenciál paraméterek szövegesen nem, de a grafikon igen:
    sections.append(widgets.HTML("<b>Potenciál (grafikon)</b>"))
    sections.append(pot_plot_output)
    if disp_items:
        sections.append(widgets.HTML("<b>Diszperzió paraméterek</b>"))
        sections.append(widgets.VBox(disp_items))
        sections.append(disp_plot_output)
    # Egyéb paraméterek nem jelennek meg
    if not sections:
        sections.append(widgets.Label("Nincs beolvasott paraméter."))
    return widgets.VBox(sections)

param_tab_output = widgets.Output()

def update_param_tab(*args):
    with param_tab_output:
        clear_output(wait=True)
        display(create_param_widgets())

# Fájl beolvasás után frissítsd a paraméter tabot
file_load_button.on_click(lambda b: update_param_tab())

param_tab = widgets.VBox([
    widgets.Label("Beolvasott paraméterek (strukturált, plotokkal):"),
    param_tab_output
])
update_param_tab()

In [3]:
# --- Quasi Particles tab ---
quasiparticle_list = []
quasiparticle_labels = []
selected_quasiparticles = []

def extract_quasiparticles_from_hdf5():
    global quasiparticle_list, quasiparticle_labels
    quasiparticle_list = []
    quasiparticle_labels = []
    path = get_hdf5_path()
    if not os.path.exists(path):
        return
    try:
        with h5py.File(path, "r") as f:
            if "scatter" in f:
                grp = f["scatter"]
                # ÚJ: Ha van "quasiparticles" csoport, abból olvasunk
                if "quasiparticles" in grp:
                    qp_group = grp["quasiparticles"]
                    for i in range(len(qp_group)):
                        qpg = qp_group[str(i)]
                        qp = {}
                        # Alap mezők
                        for key in qpg:
                            if isinstance(qpg[key], h5py.Group):
                                # dict típusú érték (pl. state)
                                subg = qpg[key]
                                subdict = {}
                                for sk in subg:
                                    subdict[sk] = subg[sk][()]
                                # attribútumokat is visszatöltjük, ha vannak
                                for sk in subg.attrs:
                                    subdict[sk] = subg.attrs[sk]
                                qp[key] = subdict
                            else:
                                qp[key] = qpg[key][()]
                        # attribútumokat is visszatöltjük, ha vannak
                        for key in qpg.attrs:
                            qp[key] = qpg.attrs[key]
                        # Index, k, energy, transmission, reflection, hullámfüggvény
                        qp_obj = {
                            "index": i,
                            "k": float(qp.get("k", 0)),
                            "energy": float(qp.get("energy", 0)),
                            "transmission": float(qp.get("transmission", 0)),
                            "reflection": float(qp.get("reflection", 0)),
                            # Hullámfüggvény: lehet "state" dict vagy "wavefunction" kulcs
                            "wavefunction": None
                        }
                        # Hullámfüggvény keresése
                        if "wavefunction" in qp:
                            qp_obj["wavefunction"] = qp["wavefunction"]
                        elif "state" in qp and isinstance(qp["state"], dict) and "value" in qp["state"]:
                            qp_obj["wavefunction"] = qp["state"]["value"]
                        quasiparticle_list.append(qp_obj)
                        quasiparticle_labels.append(
                            f"#{i} | E={qp_obj['energy']:.3e} | k={qp_obj['k']:.3e} | T={qp_obj['transmission']:.2f} | R={qp_obj['reflection']:.2f}"
                        )
                else:
                    # Régi formátum: csak vektorok
                    k_vals = grp["k_vals"][:] if "k_vals" in grp else []
                    e_vals = grp["e_vals"][:] if "e_vals" in grp else []
                    t_vals = grp["t_vals"][:] if "t_vals" in grp else []
                    r_vals = grp["r_vals"][:] if "r_vals" in grp else []
                    wf = grp["wavefunction"][:] if "wavefunction" in grp else None
                    for i in range(len(k_vals)):
                        qp = {
                            "index": i,
                            "k": k_vals[i],
                            "energy": e_vals[i],
                            "transmission": t_vals[i],
                            "reflection": r_vals[i],
                            "wavefunction": wf[i] if wf is not None and len(wf.shape) > 1 else None
                        }
                        quasiparticle_list.append(qp)
                        quasiparticle_labels.append(
                            f"#{i} | E={e_vals[i]:.3e} | k={k_vals[i]:.3e} | T={t_vals[i]:.2f} | R={r_vals[i]:.2f}"
                        )
    except Exception as e:
        pass

def update_quasiparticle_tab(*args):
    extract_quasiparticles_from_hdf5()
    with quasiparticle_tab_output:
        clear_output(wait=True)
        if not quasiparticle_list:
            display(widgets.Label("Nincs beolvasott quasiparticle adat a fájlban."))
            return
        # Többszörös kiválasztás
        select_all_checkbox = widgets.Checkbox(value=False, description="Összes kijelölése")
        multi_select = widgets.SelectMultiple(
            options=[(label, i) for i, label in enumerate(quasiparticle_labels)],
            value=(),
            description="Quasiparticles:",
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px', height='200px')
        )
        info_output = widgets.Output()

        def on_select_all_change(change):
            if change["new"]:
                multi_select.value = tuple(range(len(quasiparticle_list)))
            else:
                multi_select.value = ()

        def on_multi_select_change(change):
            global selected_quasiparticles
            selected_quasiparticles = [quasiparticle_list[i] for i in multi_select.value]
            with info_output:
                clear_output(wait=True)
                if not selected_quasiparticles:
                    print("Nincs kiválasztva quasiparticle.")
                else:
                    print(f"Kiválasztott {len(selected_quasiparticles)} db quasiparticle:")
                    for qp in selected_quasiparticles:
                        print(f"  index={qp['index']}  E={qp['energy']:.3e}  k={qp['k']:.3e}  T={qp['transmission']:.2f}  R={qp['reflection']:.2f}")

        select_all_checkbox.observe(on_select_all_change, names="value")
        multi_select.observe(on_multi_select_change, names="value")
        # Alapértelmezett: semmi nincs kiválasztva
        selected_quasiparticles.clear()
        display(widgets.VBox([
            widgets.Label("Quasiparticle lista (többszörös kijelölés lehetséges):"),
            select_all_checkbox,
            multi_select,
            info_output
        ]))

quasiparticle_tab_output = widgets.Output()
update_quasiparticle_tab()

# Fájl beolvasás után frissítjük a quasiparticle tabot is
file_load_button.on_click(lambda b: update_quasiparticle_tab())

quasiparticle_tab = widgets.VBox([
    widgets.Label("Quasi Particles (többszörös kijelölés, későbbi felhasználásra):"),
    quasiparticle_tab_output
])

In [4]:
# --- Mask fül: térbeli mask beállítása és kapcsolása ---

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Alapértelmezett paraméterek
default_uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
default_uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
default_uxgrid_width = default_uxgrid_num * default_uxgrid_dx
default_uxgrid = np.arange(-default_uxgrid_width/2, default_uxgrid_width/2, default_uxgrid_dx)

# Mask paraméter widgetek
mask_enable_checkbox = widgets.Checkbox(value=False, description="Mask bekapcsolása")
mask_ranges_text = widgets.Text(
    value="-121,-119;119,121",
    description="Kiemelt tartomány(ok):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
mask_points_widget = widgets.IntText(
    value=1024,
    description="Pontok száma (redu.):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
mask_plot_output = widgets.Output()

# Globális mask változó
current_mask = np.ones(default_uxgrid_num, dtype=bool)

def parse_ranges(ranges_str):
    """Pl. '-121,-119;119,121' -> [(-121, -119), (119, 121)]"""
    try:
        ranges = []
        for part in ranges_str.split(';'):
            if not part.strip():
                continue
            start, end = map(float, part.strip().split(','))
            ranges.append((start, end))
        return ranges
    except Exception:
        return []

def update_mask_plot(*args):
    global current_mask
    uxgrid_num = int(loaded_params.get('uxgrid_num', default_uxgrid_num))
    uxgrid_dx = float(loaded_params.get('uxgrid_dx', default_uxgrid_dx))
    uxgrid_width = uxgrid_num * uxgrid_dx
    uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
    desired_points = mask_points_widget.value
    step_size = max(1, uxgrid_num // max(1, desired_points))
    mask = np.zeros(uxgrid_num, dtype=bool)
    mask[::step_size] = True

    highlight_ranges = parse_ranges(mask_ranges_text.value)
    mask_highlight_ranges = np.zeros(uxgrid_num, dtype=bool)
    for start, end in highlight_ranges:
        mask_highlight_ranges |= (uxgrid >= start) & (uxgrid <= end)
    mask = mask | mask_highlight_ranges

    # Mask ki/bekapcsolása
    if mask_enable_checkbox.value:
        current_mask = mask
    else:
        current_mask = np.ones(uxgrid_num, dtype=bool)

    with mask_plot_output:
        clear_output(wait=True)
        plt.figure(figsize=(7,2.5))
        plt.plot(uxgrid, np.zeros_like(uxgrid), 'o', markersize=2, label='Eredeti uxgrid')
        plt.plot(uxgrid[current_mask], np.zeros_like(uxgrid[current_mask]), 'o', markersize=4, label='Aktív mask')
        for start, end in highlight_ranges:
            plt.axvspan(start, end, color='orange', alpha=0.2)
        plt.xlabel('uxgrid')
        plt.ylabel('Value')
        plt.title('Mask vizualizáció')
        plt.legend()
        plt.show()
        print(f"Maskolt pontok száma: {np.sum(current_mask)} / {uxgrid_num}")

# Widgetek eseménykezelői
mask_enable_checkbox.observe(update_mask_plot, names='value')
mask_ranges_text.observe(update_mask_plot, names='value')
mask_points_widget.observe(update_mask_plot, names='value')

# Első frissítés
update_mask_plot()

mask_tab = widgets.VBox([
    widgets.Label("Térbeli mask beállítása:"),
    mask_enable_checkbox,
    widgets.HBox([mask_points_widget, mask_ranges_text]),
    mask_plot_output
])

In [5]:
# --- Laser tab: külső lézertér paraméterek és vizualizáció ---
import matplotlib.pyplot as plt
from quantum_toolkit import potentials as pots

# Alapértelmezett értékek
default_spot_size_nm = 800
default_field_strength_gvm = 1.0
default_central_wavelength_nm = 800
default_num_of_cycles = 5
default_beta = 0.2

laser_spot_size_widget = widgets.FloatText(
    value=default_spot_size_nm,
    description="Spot méret [nm]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_field_strength_widget = widgets.FloatText(
    value=default_field_strength_gvm,
    description="Térerősség [GV/m]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_wavelength_widget = widgets.FloatText(
    value=default_central_wavelength_nm,
    description="Hullámhossz [nm]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_cycles_widget = widgets.IntText(
    value=default_num_of_cycles,
    description="Ciklusszám:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_beta_widget = widgets.FloatText(
    value=default_beta,
    description="Béta:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)

laser_plot_output = widgets.Output()

def update_laser_plot(*args):
    with laser_plot_output:
        clear_output(wait=True)
        try:
            # Paraméterek
            spot_size_nm = laser_spot_size_widget.value
            field_strength_gvm = laser_field_strength_widget.value
            central_wavelength_nm = laser_wavelength_widget.value
            num_of_cycles = laser_cycles_widget.value
            beta = laser_beta_widget.value

            # Grid paraméterek
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)

            # Unit rendszer (ha van)
            try:
                import atomic_units as au
                hb = float(loaded_params.get('hbar', 1.0))
                m0 = float(loaded_params.get('m0', 8.8))
                kappa0 = float(loaded_params.get('kappa0', 2.3))
                e0 = float(loaded_params.get('e0', 3.4))
                unit_sys = au.AtomicUnitSystem(xh=hb, xe=e0, xm=m0, xk=kappa0)
            except Exception:
                unit_sys = None

            # Lézertér paraméterek átváltása
            if unit_sys is not None:
                central_wavelength = central_wavelength_nm * unit_sys.convert_length_from("nm")
                s_min = -0.5 * spot_size_nm * unit_sys.convert_length_from("nm")
                s_max = 0.5 * spot_size_nm * unit_sys.convert_length_from("nm")
                magnitude = field_strength_gvm * unit_sys.convert_electric_field_from("GV/m")
                omega = 2 * np.pi * unit_sys.speed_of_light / central_wavelength_nm
                t_min = 0
                t_max = num_of_cycles * 2 * np.pi / omega
                utgrid_dt = 0.05
                utgrid = np.arange(t_min, t_max, utgrid_dt)
                # Átváltások
                uxgrid_nm = uxgrid * unit_sys.length_unit.to('nm').magnitude
                tvals_fs = utgrid * unit_sys.time_unit.to('fs').magnitude
                field_unit_gvm = unit_sys.electric_field_unit.to('GV/m').magnitude
            else:
                # fallback: minden SI-ben
                s_min = -0.5 * spot_size_nm
                s_max = 0.5 * spot_size_nm
                magnitude = field_strength_gvm
                t_min = 0
                t_max = num_of_cycles * 2 * np.pi / (3e8 / (central_wavelength_nm * 1e-9))
                utgrid_dt = 0.05
                utgrid = np.arange(t_min, t_max, utgrid_dt)
                uxgrid_nm = uxgrid
                tvals_fs = utgrid
                field_unit_gvm = 1.0

            # Lézertér objektum
            try:
                laserpot = pots.SmoothLaserPotential(
                    uxgrid,
                    spatial_min=s_min, spatial_max=s_max,
                    temporal_max=t_max, temporal_min=t_min,
                    noc=num_of_cycles,
                    amplitude=magnitude,
                    beta=beta
                )
            except Exception as e:
                print("Hiba a lézertér objektum létrehozásakor:", e)
                return

            # Plot: térbeli profil egy adott időpillanatban
            plt.figure(figsize=(6,2.5))
            plt.plot(
                uxgrid_nm,
                laserpot(laserpot.temporal_width/2) * field_unit_gvm,
                label="Laser (térbeli profil)"
            )
            plt.xlabel("x [nm]")
            plt.ylabel("Térerő [GV/m]")
            plt.title("Lézertér térbeli profil (idő közepén)")
            plt.grid()
            plt.legend()
            plt.tight_layout()
            plt.show()

            # Plot: időbeli profil egy adott helyen (x=0)
            plt.figure(figsize=(6,2.5))
            x0_idx = np.abs(uxgrid).argmin()
            yvals = [laserpot.value_at(uxgrid[x0_idx], t) * field_unit_gvm for t in utgrid]
            plt.plot(tvals_fs, yvals, label="Laser (időbeli profil, x=0)")
            plt.xlabel("idő [fs]")
            plt.ylabel("Térerő [GV/m]")
            plt.title("Lézertér időbeli profil (x=0)")
            plt.grid()
            plt.legend()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Lézer plot hiba:", e)

for w in [laser_spot_size_widget, laser_field_strength_widget, laser_wavelength_widget, laser_cycles_widget, laser_beta_widget]:
    w.observe(update_laser_plot, names='value')

update_laser_plot()

laser_tab = widgets.VBox([
    widgets.Label("Külső lézertér paraméterei:"),
    widgets.HBox([laser_spot_size_widget, laser_field_strength_widget]),
    widgets.HBox([laser_wavelength_widget, laser_cycles_widget, laser_beta_widget]),
    laser_plot_output
])

In [6]:
# --- VCAP fül: paraméterek és automatikus vizualizáció ---
import ipywidgets as widgets
from IPython.display import display, clear_output

# Csak a szükséges paraméter widgetek
vcap_x0_widget = widgets.FloatText(
    value=350.0,
    description="param_x0:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
vcap_lambda0_widget = widgets.FloatText(
    value=0.05,
    description="param_lambda0:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
vcap_plot_output = widgets.Output()

def update_vcap_plots(*args):
    with vcap_plot_output:
        clear_output(wait=True)
        try:
            import matplotlib.pyplot as plt
            import numpy as np
            import quantum_toolkit as quat
            # uxgrid paraméterek
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            # paraméterek
            param_x0 = vcap_x0_widget.value
            param_lambda0 = vcap_lambda0_widget.value
            v_cap = quat.vcap_generator(uxgrid, param_x0=param_x0, param_lambda0=param_lambda0)
            # 1. plot: abs és angle
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.abs(v_cap[0]), label="|Vcap[0]|")
            plt.plot(uxgrid, np.angle(v_cap[0]), label="Angle{Vcap[0]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[0]: abs és angle")
            plt.tight_layout()
            plt.show()
            # 2. plot: Re/Im v_cap[1]
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.real(v_cap[1]), label="Re{Vcap[1]}")
            plt.plot(uxgrid, np.imag(v_cap[1]), label="Im{Vcap[1]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[1]: Re és Im")
            plt.tight_layout()
            plt.show()
            # 3. plot: Re/Im v_cap[2]
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.real(v_cap[2]), label="Re{Vcap[2]}")
            plt.plot(uxgrid, np.imag(v_cap[2]), label="Im{Vcap[2]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[2]: Re és Im")
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("VCAP plot hiba:", e)

# Paraméter változásra automatikus frissítés
vcap_x0_widget.observe(update_vcap_plots, names='value')
vcap_lambda0_widget.observe(update_vcap_plots, names='value')

update_vcap_plots()

vcap_tab = widgets.VBox([
    widgets.Label("VCAP paraméterek és vizualizáció:"),
    widgets.HBox([vcap_x0_widget, vcap_lambda0_widget]),
    vcap_plot_output
])

In [7]:
# --- Compute tab: időfejlesztés kiválasztott quasiparticle-ökre ---
import ipywidgets as widgets
from IPython.display import display, clear_output
import h5py
import numpy as np
import matplotlib.pyplot as plt
import quantum_toolkit as quat
from scipy import integrate

compute_output = widgets.Output()
compute_button = widgets.Button(description="Időfejlesztés futtatása", button_style='danger')

# Segédosztály a grid attribútumhoz
class SimplePotential:
    def __init__(self, values, grid):
        self.values = values
        self.grid = grid
    def __getitem__(self, idx):
        # Támogatja az összes numpy indexelést (pl. szelet, tömb, int, numpy array, ...)
        if isinstance(idx, (int, slice)):
            return np.asarray(self.values)[idx]
        elif isinstance(idx, (np.ndarray, list)):
            return np.asarray(self.values)[np.asarray(idx)]
        else:
            raise TypeError(f"Nem támogatott index típus: {type(idx)}")
    def __array__(self):
        return np.asarray(self.values)

def run_time_evolution(b):
    with compute_output:
        clear_output(wait=True)
        if not selected_quasiparticles:
            print("Nincs kiválasztva quasiparticle a Compute futtatáshoz.")
            return
        # Paraméterek beolvasása
        try:
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            # Lézertér paraméterek
            # (A Laser tabban használt paraméterek)
            spot_size_nm = laser_spot_size_widget.value
            field_strength_gvm = laser_field_strength_widget.value
            central_wavelength_nm = laser_wavelength_widget.value
            num_of_cycles = laser_cycles_widget.value
            beta = laser_beta_widget.value
            # Időrács
            t_start = 0
            t_stop = num_of_cycles * 2 * np.pi / (3e8 / (central_wavelength_nm * 1e-9))
            utgrid_dt = 0.05
            utgrid = np.arange(t_start, t_stop, utgrid_dt)
            # Modellpotenciál (egyszerűsített)
            wallwidth = float(loaded_params.get('wallwidth', 3.0))
            wallheight = float(loaded_params.get('wallheight', 1.0))
            wallrise = float(loaded_params.get('wallrise', 1.0))
            wellwidth = float(loaded_params.get('wellwidth', 10.0))
            welldepth = float(loaded_params.get('welldepth', 1.0))
            wellfall = float(loaded_params.get('wellfall', 1.0))
            num_cells = int(loaded_params.get('num_cells', 1))
            cell_spacing = float(loaded_params.get('cell_spacing', 0.0))
            pot = np.zeros_like(uxgrid)
            cell_length = 2*wallwidth + wellwidth + cell_spacing
            for i in range(num_cells):
                offset = (i - (num_cells-1)/2) * cell_length
                left = offset - (wellwidth/2 + wallwidth)
                right = offset + (wellwidth/2 + wallwidth)
                well_left = offset - wellwidth/2
                well_right = offset + wellwidth/2
                pot[(uxgrid >= left) & (uxgrid < well_left)] = wallheight
                pot[(uxgrid > well_right) & (uxgrid <= right)] = wallheight
                pot[(uxgrid >= well_left) & (uxgrid <= well_right)] = -welldepth
            modelpot = SimplePotential(pot, uxgrid)  # <-- Itt csomagoljuk be
            # Lézertér objektum (egyetlen pulse, a Laser tab paramétereivel)
            from quantum_toolkit import potentials as pots
            s_min = -0.5 * spot_size_nm
            s_max = 0.5 * spot_size_nm
            t_max = t_stop
            laserpot = pots.SmoothLaserPotential(
                uxgrid,
                spatial_min=s_min, spatial_max=s_max,
                temporal_max=t_max, temporal_min=t_start,
                noc=num_of_cycles,
                amplitude=field_strength_gvm,
                beta=beta
            )
            laser_potentials = [laserpot]
        except Exception as e:
            print("Paraméter beolvasási hiba:", e)
            return

        # Kimeneti HDF5 fájl
        h5_path = get_hdf5_path()
        try:
            h5f = h5py.File(h5_path, "a")
        except Exception as e:
            print(f"Hiba a HDF5 fájl megnyitásakor: {e}")
            return

        run_num = 1
        for qp in selected_quasiparticles:
            kin = qp["k"]
            ein = qp["energy"]
            # Hullámfüggvény (ha van)
            psi0 = qp.get("wavefunction", None)
            if psi0 is None:
                print(f"Quasiparticle #{qp['index']} hullámfüggvény hiányzik, kihagyva.")
                continue
            for lp in laser_potentials:
                print(f"{run_num}. futás:")
                print(f"- energia: {ein}")
                print(f"- hullámszám: {kin}")
                print(f"- pulse width: {lp.temporal_width}")
                print(f"- pulse omega0: {getattr(lp, 'omega0', 'n.a.')}")
                try:
                    # Időfejlesztés
                    psite = quat.SplitTimeEvolutionCalculator(
                        psi0_omega=ein/1.0,  # hbar=1
                        psi0_initial=psi0,
                        scalarpot=modelpot,  # <-- Itt már SimplePotential példány
                        vectorpot=lp,
                        dt=utgrid_dt, t_start=t_start, t_stop=t_stop
                    )
                    psite.run()
                except Exception as e:
                    print(f"Hiba a futtatás során: {e}")
                    continue
                # Eredmény mentése HDF5-be
                grpname = f"compute/run_{run_num}"
                if grpname in h5f:
                    del h5f[grpname]
                grp = h5f.create_group(grpname)
                grp.attrs["energy"] = ein
                grp.attrs["k"] = kin
                grp.create_dataset("psi_time_evolution", data=psite.psi1timeevolution)
                grp.create_dataset("uxgrid", data=uxgrid)
                grp.create_dataset("utgrid", data=utgrid)
                # Plot: valószínűségi sűrűség
                X, Y = np.meshgrid(uxgrid, utgrid)
                Z = np.transpose(quat.probability_density(psite.psi1timeevolution))
                plt.figure(figsize=(6,3))
                pcm = plt.pcolormesh(X, Y, Z, cmap='bwr')
                plt.colorbar(pcm)
                plt.title(f'Prob. dens. (run {run_num})')
                plt.xlabel("space")
                plt.ylabel("time")
                plt.xlim(-15, 15)
                plt.ylim(0, 2*lp.temporal_width)
                plt.show()
                # Áram plot
                pc = quat.probability_current(psite.psitimeevolution)
                left_index = 0
                right_index = -1
                pc_left = pc[left_index, :]
                pc_right = pc[right_index, :]
                plt.figure()
                plt.plot(utgrid, pc_left, label="current left")
                plt.plot(utgrid, pc_right, label="current right")
                plt.xlabel("time")
                plt.ylabel("current")
                plt.legend()
                plt.grid()
                plt.show()
                # Töltés plot
                charge_left = integrate.cumtrapz(pc_left, utgrid, initial=0)
                charge_right = integrate.cumtrapz(pc_right, utgrid, initial=0)
                plt.figure()
                plt.plot(utgrid, charge_left, label="charge left")
                plt.plot(utgrid, charge_right, label="charge right")
                plt.xlabel("time")
                plt.ylabel("charge")
                plt.legend()
                plt.grid()
                plt.show()
                run_num += 1
        h5f.close()
        print("Compute futás befejezve, eredmények elmentve a HDF5 fájlba.")

compute_button.on_click(run_time_evolution)

compute_tab = widgets.VBox([
    widgets.Label("Időfejlesztés kiválasztott quasiparticle-ökre (Compute):"),
    compute_button,
    compute_output
])

In [8]:
# --- Tabs összerakása ---
tabs = widgets.Tab(children=[
    file_tab,
    param_tab,
    mask_tab,
    quasiparticle_tab,
    laser_tab,
    vcap_tab,
    compute_tab  # ÚJ: Compute fül hozzáadása
])
tab_titles = [
    'Fájl műveletek', 'Paraméterek', 'Mask', 'Quasi Particles', 'Laser', 'VCAP', 'Compute'
]
for i, title in enumerate(tab_titles):
    tabs.set_title(i, title)

display(tabs)

In [10]:
print(selected_quasiparticles[0])

{'index': 0, 'k': 0.0, 'energy': 0.40847173095515105, 'transmission': 4.691404286499328e-117, 'reflection': 1.0000000000002716, 'wavefunction': array([6.49800185e-01-2.39894929e-01j, 7.66256115e-01-2.82888433e-01j,
       8.79269111e-01-3.24610866e-01j, ...,
       4.56451662e-59-5.10678283e-59j, 4.89638375e-59-4.78951658e-59j,
       5.20625050e-59-4.45073012e-59j])}
